# AI Experiment Control Center — Kaggle Instrumentation Example

> **Important:** **Do not change your hyperparameters; these cells only observe.**
> Add these minimal cells to your existing Kaggle training notebook to stream metrics, GPU memory telemetry, heartbeats, and redacted live console logs (`ecc_console.log`) to your W&B workspace and ECC dashboard.

## 1. Install `experiment-monitor`

Option A installs directly from your GitHub repository (replace `<YOUR_GH_USER>`). Option B works offline if you upload the `monitor/` folder as a private Kaggle Dataset.

In [ ]:
# [Optional] Option A — only if the repo is on GitHub:
# Option A: Install directly from GitHub (replace <YOUR_GH_USER>)
# !pip install -q "git+https://github.com/<YOUR_GH_USER>/ai-experiment-control-center.git#subdirectory=monitor"

In [ ]:
# Option B (default): install the wheel from your private Kaggle Dataset `experiment-monitor`
# Attach it first: notebook sidebar -> Add Input -> Your Datasets -> experiment-monitor.
import glob, subprocess, sys
wheels = sorted(glob.glob("/kaggle/input/**/experiment_monitor-*.whl", recursive=True))
assert wheels, "Wheel not found: attach the 'experiment-monitor' dataset via Add Input"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", wheels[-1]])
print("Installed", wheels[-1])

## 2. Authenticate W&B from Kaggle Secrets

1. In the Kaggle notebook editor, open **Add-ons → Secrets**.
2. Click **Add Secret**, set **Label** = `WANDB_API_KEY` and **Value** = your personal W&B API key (from `https://wandb.ai/authorize`).
3. Toggle **Attach to notebook** ON for `WANDB_API_KEY`.

`em.login_from_notebook_secrets("WANDB_API_KEY")` reads the secret via `kaggle_secrets.UserSecretsClient` (falling back to Colab Secrets or the `WANDB_API_KEY` env var) and never prints or returns the key.

In [ ]:
import experiment_monitor as em

ok = em.login_from_notebook_secrets("WANDB_API_KEY")
print("W&B authentication:", "OK" if ok else "FAILED — check Add-ons -> Secrets -> WANDB_API_KEY")

## 3. Initialize ECC Monitor (`em.init`)

Keep your existing config dictionary untouched; pass it to `config=existing_config_dict` along with ECC metadata so the dashboard can link this W&B run to your Kaggle account (`kaggle-1`) and kernel (`<owner>/<slug>`).

In [ ]:
existing_config_dict = {
    "learning_rate": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "batch_size": 8,
    "epochs": 3,
}

# Set INIT_STANDALONE = True when running this cell against your real W&B project
INIT_STANDALONE = False
if INIT_STANDALONE:
    mon = em.init(
        project="shared-task-4",
        name="qwen3-vl-lora-r16-kaggle1",
        account="kaggle-1",
        notebook_url="https://www.kaggle.com/code/<owner>/<slug>",
        kaggle_kernel="<owner>/<slug>",
        model_name="Qwen/Qwen3-VL-4B-Instruct",
        seed=42,
        dataset_version="v2",
        dataset_split="train/val=90/10 seed42",
        prompt_strategy="vi-default",
        eval_signature="v2-vi-90-10-seed42",
        config=existing_config_dict,
    )

## 4A. Variant A — HuggingFace `Trainer` / TRL `SFTTrainer`

- **Do not change your hyperparameters; these cells only observe.**
- Attach `make_hf_callback(mon)` via `trainer.add_callback(make_hf_callback(mon))`. Because TRL `SFTTrainer` subclasses `transformers.Trainer`, the exact same callback works for both.
- **`report_to` note**: Set `report_to="none"` in `TrainingArguments` / `SFTConfig` (recommended) so ECC logs metrics once under standardized `train/*` and `eval/*` keys. If you already use `report_to="wandb"`, `em.init(...)` automatically reuses the active `wandb.run` without opening a second run.

In [ ]:
from experiment_monitor import make_hf_callback

USE_HF_TRAINER = False
if USE_HF_TRAINER:
    # Assume `trainer` is your existing HuggingFace Trainer or TRL SFTTrainer
    # configured with report_to="none" (recommended) and all your original hyperparameters.
    mon.track_best("eval/macro_f1", mode="max")
    trainer.add_callback(make_hf_callback(mon))
    with mon:
        trainer.train()

## 4B. Variant B — Plain PyTorch Loop (CPU-only Smoke Test)

- **Do not change your hyperparameters; these cells only observe.**
- Wrap your training loop in `with mon:` so uncaught exceptions record `ecc/end_reason="exception"`, `ecc/exception_type`, redacted `ecc/exception_message`, and flush the traceback to `ecc_console.log`.
- Below is a tiny 5-step CPU-only dummy loop (`DRY_RUN = True`) with random tensors so you can verify logging end-to-end without a GPU and without touching your real training code.

In [ ]:
# ECC-SMOKE
import os
import tempfile
import torch
import experiment_monitor as em

DRY_RUN = True
if DRY_RUN:
    torch.manual_seed(42)
    existing_config_dict = {
        "learning_rate": 1e-2,
        "batch_size": 4,
        "epochs": 1,
    }
    total_steps = 5
    model = torch.nn.Linear(8, 2)
    optimizer = torch.optim.SGD(model.parameters(), lr=existing_config_dict["learning_rate"])
    loss_fn = torch.nn.CrossEntropyLoss()

    mon = em.init(
        project="shared-task-4",
        name="smoke-pytorch-kaggle-1",
        account="kaggle-1",
        notebook_url="https://www.kaggle.com/code/<owner>/<slug>",
        kaggle_kernel="<owner>/<slug>",
        model_name="dummy-linear-cpu",
        seed=42,
        dataset_version="smoke-v0",
        total_steps=total_steps,
        total_epochs=1.0,
        config=existing_config_dict,
    )

    with tempfile.TemporaryDirectory() as ckpt_dir:
        with mon:
            mon.track_best("eval/macro_f1", mode="max")
            for step in range(1, total_steps + 1):
                x = torch.randn(4, 8)
                y = torch.randint(0, 2, (4,))
                assert x.shape == (4, 8), f"Unexpected input shape: {x.shape}"

                optimizer.zero_grad()
                logits = model(x)
                assert logits.shape == (4, 2), f"Unexpected logits shape: {logits.shape}"
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()

                epoch = step / float(total_steps)
                mon.log(
                    {
                        "train/loss": float(loss.item()),
                        "train/lr": existing_config_dict["learning_rate"],
                        "train/epoch": epoch,
                    },
                    step=step,
                )
                mon.set_progress(step=step, total_steps=total_steps, epoch=epoch)

                if step == total_steps:
                    eval_f1 = 0.75
                    mon.log({"eval/loss": float(loss.item()), "eval/macro_f1": eval_f1}, step=step)
                    ckpt_path = os.path.join(ckpt_dir, f"checkpoint-step-{step}.pt")
                    torch.save(model.state_dict(), ckpt_path)
                    mon.log_checkpoint(ckpt_path, step=step, metric=eval_f1, metric_name="eval/macro_f1")
    print("ECC Kaggle smoke test finished cleanly.")

## 5. Resume an Interrupted Run (`resume_id=`)

If a Kaggle session hits the 12-hour limit or restarts, pass the previous W&B run ID to `resume_id=` so metrics and progress continue on the same run in ECC.

In [ ]:
RUN_RESUME_EXAMPLE = False
if RUN_RESUME_EXAMPLE:
    PREVIOUS_RUN_ID = "<YOUR_WANDB_RUN_ID>"  # e.g. 8-char W&B run id from the run URL
    with em.init(
        project="shared-task-4",
        name="qwen3-vl-lora-r16-kaggle1",
        account="kaggle-1",
        notebook_url="https://www.kaggle.com/code/<owner>/<slug>",
        kaggle_kernel="<owner>/<slug>",
        model_name="Qwen/Qwen3-VL-4B-Instruct",
        seed=42,
        dataset_version="v2",
        resume_id=PREVIOUS_RUN_ID,
        config=existing_config_dict,
    ) as mon:
        # Resume your existing training loop / trainer.train(resume_from_checkpoint=True)
        pass